# **DIABETES CLASSIFICATION MODEL**

In [16]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
file_path = "diabetes.csv"
df = pd.read_csv(file_path)

In [3]:
print("Dataset Shape:", df.shape)
print("First 5 Rows:\n", df.head())
print("Missing Values:\n", df.isnull().sum())

Dataset Shape: (768, 9)
First 5 Rows:
    Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
Missing Values:
 Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome

In [4]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
models = {
    "RandomForest": RandomForestClassifier(),
    "GradientBoosting": GradientBoostingClassifier(),
    "SVM": SVC(probability=True)
}

In [8]:
param_grids = {
    "RandomForest": {"n_estimators": [100, 200, 300], "max_depth": [5, 10, None]},
    "GradientBoosting": {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [3, 5]},
    "SVM": {"C": [0.1, 1, 10], "kernel": ["rbf", "linear"]}
}

In [9]:
best_model = None
best_accuracy = 0

In [10]:
for name, model in models.items():
    grid_search = GridSearchCV(model, param_grids[name], cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train_scaled, y_train)
    best_model_params = grid_search.best_params_

    best_model_instance = grid_search.best_estimator_
    y_pred = best_model_instance.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)

    print(f" {name} Best Params: {best_model_params}")
    print(f" {name} Accuracy: {accuracy:.4f}\n")

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = best_model_instance

print("\n🏆 Best Model Selected:", type(best_model).__name__)


 RandomForest Best Params: {'max_depth': 5, 'n_estimators': 300}
 RandomForest Accuracy: 0.7532

 GradientBoosting Best Params: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 100}
 GradientBoosting Accuracy: 0.7727

 SVM Best Params: {'C': 0.1, 'kernel': 'linear'}
 SVM Accuracy: 0.7208


🏆 Best Model Selected: GradientBoostingClassifier


In [11]:
y_pred_best = best_model.predict(X_test_scaled)
print("\n🔹 Classification Report:\n", classification_report(y_test, y_pred_best))
print("\n🔹 Confusion Matrix:\n", confusion_matrix(y_test, y_pred_best))


🔹 Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.87      0.83       100
           1       0.71      0.59      0.65        54

    accuracy                           0.77       154
   macro avg       0.75      0.73      0.74       154
weighted avg       0.77      0.77      0.77       154


🔹 Confusion Matrix:
 [[87 13]
 [22 32]]


In [17]:
joblib.dump(best_model, "diabetes_model.pkl")
joblib.dump(scaler, "diabetes_scaler.pkl")


['diabetes_scaler.pkl']

In [12]:
new_data = np.array([
    [6, 148, 72, 35, 120, 33.6, 0.627, 50],
    [1, 85, 66, 29, 96, 26.6, 0.351, 31],
    [3, 120, 70, 30, 110, 32.0, 0.500, 40]
])

In [13]:
new_data_scaled = scaler.transform(new_data)


c:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [14]:
predictions = best_model.predict(new_data_scaled)


In [15]:
for i, pred in enumerate(predictions):
    result = "🟥 Detected Diabetes" if pred == 1 else "🟩 Not Detected Diabetes"
    print(f"Case {i+1}: {result}")

Case 1: 🟥 Detected Diabetes
Case 2: 🟩 Not Detected Diabetes
Case 3: 🟩 Not Detected Diabetes
